# Progetto Machine Learning - Andrea Pezzolla & Matteo Maruca
###  https://www.kaggle.com/datasets/dimahkiin/russian-license-plates-5-class-detection

Train -> Serve a determinare i param dei vari modelli

Val -> Serve a confrontare i modelli e scegliere il migliore da usare in test

Test -> Test finale con performance ecc del modello migliore scelto prima

## Step:

1. Caricamento ✅
È già fatto: df ha una riga per targa.

2. Pulizia
Togliere le righe con class_id NaN, cioè le 35 immagini senza targhe.
Togliere le bbox non valide: w o h minori o uguali a 0, oppure coordinate fuori da [0, 1].
Togliere i ritagli troppo piccoli, ad esempio sotto i 10 px di altezza, perché non contengono informazione utile.
3. Costruzione dell'input (X, y)
Per ogni riga, aprire l'immagine e ritagliare la bbox. Le coordinate YOLO vanno convertite in pixel.
Ridimensionare ogni ritaglio a una misura fissa. Senza EDA la puoi giustificare con la targa russa standard, che misura 520×112 mm (rapporto circa 4.6:1). Quindi ad esempio 64×14 RGB.
Tenere i colori RGB: blu per polizia, rosso per diplomatiche, giallo per taxi, nero per militari. Il colore è l'informazione più utile per distinguere le classi.
Appiattire il ritaglio come nel lab: 64·14·3 = 2688 feature per targa.
y è class_id, un intero da 0 a 4, che è il formato giusto per SparseCategoricalCrossentropy.
Usare lo split già fornito per ottenere X_train, y_train, X_val, y_val, X_test, y_test.
Mostrare 5 o 10 ritagli con la loro etichetta. Non è EDA: serve solo a controllare che il ritaglio sia giusto.
4. Preprocessing
Normalizzare i pixel dividendo per 255, oppure con uno StandardScaler calcolato solo sul train.
Calcolare i pesi delle classi con compute_class_weight, perché i taxi sono pochi.
5. Modello 1: rete neurale Keras (come nn3)

model = Sequential([
    Dense(128, activation="relu"),
    Dense(64, activation="relu"),
    Dense(5, activation="linear"),   # 5 classi, logit
])
model.compile(loss=SparseCategoricalCrossentropy(from_logits=True),
              optimizer=Adam(1e-3), metrics=["accuracy"])
model.fit(X_train, y_train, validation_data=(X_val, y_val),
          epochs=..., class_weight=..., callbacks=[EarlyStopping(...)])
Tuning: una griglia a mano su numero di neuroni, numero di layer, learning rate ed eventualmente dropout. Il modello migliore si sceglie con l'F1 macro sul validation set.
Come nel lab, le probabilità si ottengono con tf.nn.softmax(model.predict(X)). Servono per la curva ROC.

6. Modello 2: sklearn con Pipeline e GridSearchCV
Pipeline([StandardScaler, PCA, MLPClassifier]) oppure RandomForestClassifier.
Per la cross-validation usare GridSearchCV con StratifiedGroupKFold e groups = immagine. Così due targhe della stessa foto non finiscono in fold diversi, ed eviti il leakage.

7. Ensemble
Usare BaggingClassifier o VotingClassifier sui modelli sklearn.
In alternativa, fare la media delle softmax di più reti Keras addestrate con seed diversi.

8. Confronto sul validation set e test finale
Scegliere il modello migliore guardando i risultati sul validation set.
Valutarlo una sola volta sul test con:
accuracy;
precision, recall e F1 (macro e per classe);
matrice di confusione;
curve ROC one-vs-rest.

9. Discussione
Spiegare quali classi vengono confuse. Mi aspetto problemi soprattutto con taxi e civilian.
Spiegare cosa faresti con più tempo: una CNN (Conv2D) invece dei soli Dense, data augmentation, più esempi di taxi.


## Imports

In [3]:
# --- Librerie standard ---
import os
import json
import random
from pathlib import Path

# --- Manipolazione dati ---
import numpy as np
import pandas as pd
import yaml
from PIL import Image

# --- Visualizzazione ---
import matplotlib.pyplot as plt
import matplotlib.patches as patches

# --- Scikit-learn: preprocessing, modelli, tuning, ensemble, metriche ---
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, StratifiedKFold
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import BaggingClassifier, VotingClassifier, RandomForestClassifier
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    roc_curve, auc, RocCurveDisplay,
)
from sklearn.preprocessing import label_binarize

# --- Deep Learning (reti neurali) con TensorFlow / Keras ---
import tensorflow as tf
import keras
from keras import layers, models, callbacks, optimizers

# --- Utility ---
from tqdm import tqdm

print("TensorFlow:", tf.__version__)
print("Keras:", keras.__version__)
print("GPU disponibili:", tf.config.list_physical_devices("GPU"))

TensorFlow: 2.22.0-rc0
Keras: 3.16.0.dev2026102604
GPU disponibili: []


In [4]:
# Fissiamo il seed per rendere gli esperimenti riproducibili
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)

## Caricamento del dataset

Il dataset è in formato YOLO (object detection): per ogni immagine c'è un file `.txt` con una riga per targa (classe + bounding box). 
Lo split train / val / test è già fornito. Carichiamo tutte le annotazioni in un DataFrame; la pulizia verrà fatta nella sezione successiva.

In [5]:
# Percorsi del dataset (formato YOLO: images/<split>/*.jpg + labels/<split>/*.txt)
DATASET_DIR = Path("dataset")
SPLITS = ["train", "val", "test"]

# Il file yaml contiene il mapping id -> nome della classe
with open(DATASET_DIR / "data_multiclass.yaml") as f:
    data_cfg = yaml.safe_load(f)

CLASS_NAMES = data_cfg["names"]   # {0: 'civilian', 1: 'taxi', ...}
NUM_CLASSES = data_cfg["nc"]
print(NUM_CLASSES, "classi:", CLASS_NAMES)

5 classi: {0: 'civilian', 1: 'taxi', 2: 'military', 3: 'police', 4: 'diplomatic'}


In [6]:
def load_split(split):
    """Legge tutte le annotazioni di uno split e restituisce un DataFrame.

    Ogni file .txt ha una riga per targa nel formato YOLO:
        <class_id> <x_center> <y_center> <width> <height>   (coordinate normalizzate in [0, 1])
    Le immagini senza targhe hanno un file vuoto: le teniamo con class_id = NaN
    così da poterle gestire esplicitamente nella fase di pulizia.
    """
    rows = []
    img_dir = DATASET_DIR / "images" / split
    lbl_dir = DATASET_DIR / "labels" / split
    for img_path in sorted(img_dir.glob("*.jpg")):
        lbl_path = lbl_dir / (img_path.stem + ".txt")
        lines_ = lbl_path.read_text().strip().splitlines() if lbl_path.exists() else []
        if not lines_:
            rows.append({"split": split, "image": str(img_path), "class_id": np.nan,
                         "x_c": np.nan, "y_c": np.nan, "w": np.nan, "h": np.nan})
        for line in lines_:
            c, x, y, w, h = line.split()
            rows.append({"split": split, "image": str(img_path), "class_id": int(c),
                         "x_c": float(x), "y_c": float(y), "w": float(w), "h": float(h)})
    return pd.DataFrame(rows)

# Un'unica tabella con tutte le annotazioni (una riga = una targa)
df = pd.concat([load_split(s) for s in SPLITS], ignore_index=True)
df["class_id"] = df["class_id"].astype("Int64")
df["class_name"] = df["class_id"].map(CLASS_NAMES)

print(df.shape)
df.head()

(3173, 8)


,split,image,class_id,x_c,y_c,w,h,class_name
0,train,dataset/images/train/0.jpg,0,0.503750,0.681875,0.324167,0.116250,civilian
1,train,dataset/images/train/0.jpg,0,0.246667,0.335000,0.165000,0.065000,civilian
2,train,dataset/images/train/1.jpg,0,0.800347,0.514815,0.149306,0.070370,civilian
3,train,dataset/images/train/10.jpg,0,0.507031,0.607292,0.301563,0.106250,civilian
4,train,dataset/images/train/100.jpg,0,0.513889,0.720741,0.292222,0.099259,civilian


In [8]:
# Prima panoramica: numero di targhe per classe in ciascuno split (NaN = immagini senza targhe)
pd.crosstab(df["class_name"].fillna("<none>"), df["split"], margins=True)

split,test,train,val,All
class_name,,,,
<none>,4,31,0,35
civilian,138,1102,57,1297
diplomatic,46,364,70,480
military,63,503,105,671
police,51,402,69,522
taxi,16,123,29,168
All,318,2525,330,3173
